In [53]:
import requests
import difflib
import csv
import re
from collections import defaultdict
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

#parameters

URL = "https://en.wikipedia.org/w/api.php"

HEADERS = {
    "User-Agent": "WikiEditStudy/1.0 (student project)"
}

PARAMS = {
    "action": "query",
    "format": "json",
    "prop": "revisions",
    "titles": "Legal status of transgender people",
    "rvprop": "content",
    "rvslots": "main",
    "rvstart": "2023-12-31T23:59:59Z",
    "rvend": "2022-01-01T00:00:00Z",
    "rvlimit": "max",
    "rvdir": "older"
}


In [54]:
# tokenizer

STOP_WORDS = list(ENGLISH_STOP_WORDS) + ["date", "archive","url","web","access","https","ref","org","website","www","http"]
def tokenize(text):
    """Lowercase and extract alphabetic words."""
    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    return [w for w in words if w not in STOP_WORDS]

In [55]:
#edits finder

def get_revisions():
    revisions = []
    cont = {}

    while True:
        params = PARAMS.copy()
        params.update(cont)

        r = requests.get(URL, params=params, headers=HEADERS)
        data = r.json()

        pages = data["query"]["pages"]
        page = next(iter(pages.values()))

        for rev in page["revisions"]:
            slot = rev["slots"]["main"]
            text = slot.get("*", slot.get("content", ""))
            if text:
                revisions.append(text)

        if "continue" in data:
            cont = data["continue"]
        else:
            break

    return revisions

In [56]:
def main():
    revisions = get_revisions()

    added_counts = defaultdict(int)
    removed_counts = defaultdict(int)

    for i in range(len(revisions) - 1):
        old_words = tokenize(revisions[i])
        new_words = tokenize(revisions[i + 1])

        diff = difflib.ndiff(old_words, new_words)

        for d in diff:
            word = d[2:]
            if d.startswith("+ "):
                added_counts[word] += 1
            elif d.startswith("- "):
                removed_counts[word] += 1

    # Merge counts and compute net
    all_words = set(list(added_counts.keys()) + list(removed_counts.keys()))
    results = []

    for word in all_words:
        added = added_counts.get(word, 0)
        removed = removed_counts.get(word, 0)
        net = added - removed
        results.append({
            "word": word,
            "times_added": added,
            "times_removed": removed,
            "net_change": net
        })

    # Sort by times_added descending
    results.sort(key=lambda x: x["times_added"], reverse=True)

    # Write CSV
    with open("wiki_word_summary.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["word", "times_added", "times_removed", "net_change"])
        writer.writeheader()
        writer.writerows(results)

    print("Saved wiki_word_summary.csv")


if __name__ == "__main__":
    main()


Saved wiki_word_summary.csv


In [65]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# -----------------------------
# 1. Load Wikipedia word-change data
# -----------------------------
df = pd.read_csv("wiki_word_summary.csv")  # columns: word, times_added, times_removed, net_change

# -----------------------------
# 2. Load MPQA Subjectivity Lexicon
# -----------------------------
lex_file = "subj_lexicon.tff"  # replace with your downloaded MPQA file

lex_data = []
with open(lex_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = dict(re.findall(r"(\w+)=([^\s]+)", line))
        word = parts.get("word1")
        subj_type = parts.get("type")
        polarity = parts.get("priorpolarity")
        if word:
            lex_data.append({
                "word": word.lower(),
                "subjectivity_type": subj_type,
                "polarity": polarity
            })

mpqa_df = pd.DataFrame(lex_data)

# -----------------------------
# 3. Merge subjectivity labels into word-change data
# -----------------------------
df = df.merge(mpqa_df, on="word", how="left")
df["subjectivity_type"] = df["subjectivity_type"].fillna("none")
df["polarity"] = df["polarity"].fillna("none")

# -----------------------------
# 4. Create supervised learning labels
# -----------------------------
# Binary: is the word subjective or not?
df["is_subjective"] = (df["subjectivity_type"] != "none").astype(int)

# Only keep words with labels
df_labeled = df[df["subjectivity_type"] != "none"]

# Features
X = df_labeled[["times_added", "times_removed", "net_change"]]

# Targets
y_subjective = df_labeled["is_subjective"]
y_polarity = df_labeled["polarity"]  # positive, negative, or neutral/none

# -----------------------------
# 5. Split into train/test
# -----------------------------
X_train, X_test, y_sub_train, y_sub_test, y_pol_train, y_pol_test = train_test_split(
    X, y_subjective, y_polarity, test_size=0.3, random_state=42
)

# -----------------------------
# 6. Train Random Forest for subjectivity
# -----------------------------
clf_subjective = RandomForestClassifier(n_estimators=100, random_state=42)
clf_subjective.fit(X_train, y_sub_train)

# -----------------------------
# 7. Train Random Forest for polarity
# -----------------------------
clf_polarity = RandomForestClassifier(n_estimators=100, random_state=42)
clf_polarity.fit(X_train, y_pol_train)

# -----------------------------
# 8. Predict and evaluate subjectivity
# -----------------------------
y_sub_pred = clf_subjective.predict(X_test)
print("Classification report for SUBJECTIVITY:")
print(classification_report(y_sub_test, y_sub_pred))

# -----------------------------
# 9. Predict and evaluate polarity
# -----------------------------
y_pol_pred = clf_polarity.predict(X_test)
print("Classification report for POLARITY:")
print(classification_report(y_pol_test, y_pol_pred))

# -----------------------------
# 10. Add predictions to CSV and export
# -----------------------------
df_labeled["predicted_subjective"] = clf_subjective.predict(df_labeled[["times_added","times_removed","net_change"]])
df_labeled["predicted_polarity"] = clf_polarity.predict(df_labeled[["times_added","times_removed","net_change"]])

df_labeled.to_csv("wiki_word_subjectivity_polarity_predictions.csv", index=False)
print("Saved wiki_word_subjectivity_polarity_predictions.csv with predictions.")


Classification report for SUBJECTIVITY:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        61

    accuracy                           1.00        61
   macro avg       1.00      1.00      1.00        61
weighted avg       1.00      1.00      1.00        61

Classification report for POLARITY:
              precision    recall  f1-score   support

    negative       0.44      0.22      0.30        18
     neutral       0.33      0.07      0.11        15
    positive       0.53      0.93      0.68        28

    accuracy                           0.51        61
   macro avg       0.44      0.41      0.36        61
weighted avg       0.46      0.51      0.42        61

Saved wiki_word_subjectivity_polarity_predictions.csv with predictions.


/tmp/ipython-input-786976057.py:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_labeled["predicted_subjective"] = clf_subjective.predict(df_labeled[["times_added","times_removed","net_change"]])
/tmp/ipython-input-786976057.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_labeled["predicted_polarity"] = clf_polarity.predict(df_labeled[["times_added","times_removed","net_change"]])
